# Sybil Zero-Day — Attention-AE Baseline (complexity vs simplicity)

**Goal.** Test whether a higher-capacity, **attention-based autoencoder** overcomes the structural
limitation we identified, or whether the simple benign-anchored detector on the right representation
still wins ("simple beats complex").

**Design (fair Occam comparison), same data/protocol/seeds as the consolidated baselines:**
- **Complex + rich representation:** a tabular self-attention autoencoder (`attn_ae_pf`) trained on
  the known-manifold per-flow features.
- **Complex + right representation:** the same model on the source-rate features (`attn_ae_rate`).
- **Simple + right representation:** benign-anchored Mahalanobis on source-rate (`mahal_rate`, our method).
All under leave-one-attack-class-out (LOACO) with **source-disjoint** testing; multi-seed.

**Pre-committed reading.** If `attn_ae_*` source-disjoint ROC does NOT exceed `mahal_rate_sdj`
(\u2248 0.907), we conclude that capacity does not overcome the structural limitation: a Sybil flow
lies inside the known per-flow manifold, so even attention reconstructs it as familiar. The gain
comes from representation + benign-anchoring, not model complexity. If the attention model genuinely
wins (source-disjoint AND not artifact-driven), we report it honestly as a co-method.

> Runs on Colab (PyTorch). CPU is sufficient with the small model below; set `EPOCHS`/`SUBSAMPLE` for speed.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import torch, torch.nn as nn
from numpy.linalg import pinv
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
torch.manual_seed(42); np.random.seed(42)
DEV="cuda" if torch.cuda.is_available() else "cpu"
print("torch",torch.__version__,"| device",DEV)

torch 2.11.0+cu128 | device cuda


In [ ]:
# config + helpers (identical protocol to UAV_Sybil_FinalBaselines)
DATA="/content/data/"; SEEDS=(42,1,2,3,4); RS=42; FPR_GRID=(0.05,0.10,0.20)
EPOCHS=40        #@param
SUBSAMPLE=0      #@param
SUBSAMPLE=None if SUBSAMPLE in (0,None) else int(SUBSAMPLE)
UAV_CFG={"path":DATA+"UAVIDS-2025.csv","label_col":"label","normal":"Normal Traffic",
  "attacks":["Blackhole Attack","Flooding Attack","Sybil Attack","Wormhole Attack"],
  "leak_clean":["FlowID","SrcAddr","DstAddr","Protocol"]}
ID_CFG={"src":"SrcAddr","dst":"DstAddr"}; TARGET="Sybil Attack"
RATE=["s_fanout_rate","s_flows_per_dst","s_dst_entropy_norm"]

def load(cfg):
    df=pd.read_csv(cfg["path"],low_memory=False).reset_index(drop=True)
    if SUBSAMPLE:
        df=pd.concat([g.sample(min(len(g),SUBSAMPLE),random_state=RS) for _,g in df.groupby(cfg["label_col"])]).reset_index(drop=True)
    return df
def source_rate_features(df,idcfg):
    src=idcfg["src"]; dst=idcfg["dst"]
    s=df[src].astype(str).fillna("NA").values; dd=df[dst].astype(str).fillna("NA").values
    tmp=pd.DataFrame({"s":s,"d":dd}); g=tmp.groupby("s")
    flowcount=g["s"].transform("size").astype(float).values; fanout=g["d"].transform("nunique").astype(float).values
    ent_map={}
    for k,gg in tmp.groupby("s"):
        vc=gg["d"].value_counts().values.astype(float); p=vc/vc.sum(); ent_map[k]=float(-(p*np.log(p+1e-12)).sum())
    ent=np.array([ent_map[x] for x in s]); fc=np.clip(flowcount,1,None); fo=np.clip(fanout,1,None)
    return pd.DataFrame({"s_fanout_rate":fanout/fc,"s_flows_per_dst":flowcount/fo,
                         "s_dst_entropy_norm":ent/np.log(np.clip(fanout,2,None))}).fillna(0.0).reset_index(drop=True)
def feats_perflow(df,cfg):
    drop=set(cfg["leak_clean"])|{cfg["label_col"]}|set(RATE)
    X=df.drop(columns=[c for c in df.columns if c in drop],errors="ignore").copy()
    for c in X.columns:
        if not pd.api.types.is_numeric_dtype(X[c]): X[c]=LabelEncoder().fit_transform(X[c].astype(str))
    return X.astype(float).reset_index(drop=True)
def loaco_split(df,label_col,target,seed,val=0.3,test=0.3):
    rng=np.random.default_rng(seed)
    dft=df[df[label_col].astype(str)==target]; dfk=df[df[label_col].astype(str)!=target]
    idx=rng.permutation(len(dfk)); nv=int(len(idx)*val)
    valk=dfk.iloc[idx[:nv]]; tr=dfk.iloc[idx[nv:]]; nt=int(len(tr)*test)
    return tr.iloc[nt:], valk, pd.concat([tr.iloc[:nt],dft])
def fit_mahal(Xn):
    mu=Xn.mean(0); cov=np.cov(Xn.T)+1e-6*np.eye(Xn.shape[1]); return mu,pinv(cov)
def mahal(X,mu,P):
    d=X-mu; return np.einsum('ij,jk,ik->i',d,P,d)
def scale(A,Bv,Be):
    imp=SimpleImputer(strategy="mean").fit(A); sc=StandardScaler().fit(imp.transform(A))
    return sc.transform(imp.transform(A)),sc.transform(imp.transform(Bv)),sc.transform(imp.transform(Be))
print("helpers ok")

helpers ok


## Tabular self-attention autoencoder

Each scalar feature is projected to a token embedding and tagged with a learned feature embedding;
multi-head self-attention layers encode cross-feature dependencies; a small latent bottleneck forces
reconstruction. Trained on the known-manifold (benign + known attacks); anomaly score = reconstruction
error. This is the attention-augmented analogue of the plain AE baseline.

In [ ]:
class TabAttnAE(nn.Module):
    def __init__(self,n_features,d=32,nhead=4,nlayers=2,latent=8):
        super().__init__()
        self.n=n_features; self.d=d
        self.val_proj=nn.Linear(1,d)
        self.feat_emb=nn.Parameter(torch.randn(n_features,d)*0.02)
        enc=nn.TransformerEncoderLayer(d_model=d,nhead=nhead,dim_feedforward=2*d,batch_first=True,dropout=0.0)
        self.encoder=nn.TransformerEncoder(enc,num_layers=nlayers)
        self.to_latent=nn.Linear(n_features*d,latent)
        self.from_latent=nn.Linear(latent,n_features*d)
        self.out=nn.Linear(d,1)
    def forward(self,x):
        B=x.size(0)
        tok=self.val_proj(x.unsqueeze(-1))+self.feat_emb.unsqueeze(0)   # [B,n,d]
        z=self.encoder(tok)                                            # [B,n,d]
        lat=self.to_latent(z.reshape(B,-1))
        h=self.from_latent(lat).reshape(B,self.n,self.d)
        return self.out(h).squeeze(-1)                                 # [B,n]

def train_attn_ae(Xtr,seed,epochs=EPOCHS,bs=512,lr=1e-3):
    torch.manual_seed(seed)
    n=Xtr.shape[1]; d=32; nhead=4 if n>=4 else 1
    m=TabAttnAE(n,d=d,nhead=nhead).to(DEV)
    opt=torch.optim.Adam(m.parameters(),lr=lr)
    X=torch.tensor(Xtr,dtype=torch.float32)
    ds=torch.utils.data.TensorDataset(X)
    dl=torch.utils.data.DataLoader(ds,batch_size=bs,shuffle=True)
    lossf=nn.MSELoss()
    m.train()
    for ep in range(epochs):
        for (xb,) in dl:
            xb=xb.to(DEV); opt.zero_grad()
            rec=m(xb); loss=lossf(rec,xb); loss.backward(); opt.step()
    return m
@torch.no_grad()
def attn_err(m,X):
    m.eval(); xb=torch.tensor(X,dtype=torch.float32).to(DEV)
    rec=m(xb); return ((xb-rec)**2).mean(1).cpu().numpy()
print("model ok")

model ok


## Run: attention-AE (per-flow & source-rate) vs simple benign-Mahalanobis

In [ ]:
def op_pts(base_norm,se,is_t,is_n,ka,grid):
    out={}
    for f in grid:
        thr=np.percentile(base_norm,100*(1-f))
        out[f]=dict(det=float(np.mean(se[is_t]>thr)),fpr_normal=float(np.mean(se[is_n]>thr)),
                    fpr_known=float(np.mean(se[ka]>thr)) if ka.any() else np.nan)
    return out

def run(seeds=SEEDS,grid=FPR_GRID):
    cfg=UAV_CFG; idcfg=ID_CFG; lab=cfg["label_col"]; normal=cfg["normal"]; tgt=TARGET; src=idcfg["src"]
    df=load(cfg); df=pd.concat([df,source_rate_features(df,idcfg)],axis=1)
    roc={k:[] for k in ["attn_ae_pf_sdj","attn_ae_rate_sdj","mahal_rate_sdj"]}; op={"attn_ae_pf":[],"mahal_rate":[]}
    for seed in seeds:
        tr,valk,test_=loaco_split(df,lab,tgt,seed); test_=test_.reset_index(drop=True)
        ytr=tr[lab].astype(str).values; yv=valk[lab].astype(str).values; ye=test_[lab].astype(str).values
        is_t=(ye==tgt); is_n=(ye==normal); ka=(~is_t)&(~is_n); bmask=(ytr==normal)
        seen=is_t & test_[src].astype(str).isin(set(tr[src].astype(str))).values; sdj=~seen
        # attn-AE on per-flow
        Ap=feats_perflow(tr,cfg); Bvp=feats_perflow(valk,cfg).reindex(columns=Ap.columns,fill_value=0); Bep=feats_perflow(test_,cfg).reindex(columns=Ap.columns,fill_value=0)
        Za,Zv,Ze=scale(Ap,Bvp,Bep); m=train_attn_ae(Za,seed); e=attn_err(m,Ze)
        roc["attn_ae_pf_sdj"].append(roc_auc_score(is_t[sdj].astype(int),e[sdj]))
        op["attn_ae_pf"].append(op_pts(attn_err(m,Zv)[yv==normal],e[sdj],is_t[sdj],is_n[sdj],ka[sdj],grid))
        # attn-AE on source-rate
        Ar=tr[RATE].reset_index(drop=True); Bvr=valk[RATE].reset_index(drop=True); Ber=test_[RATE].reset_index(drop=True)
        Za,Zv,Ze=scale(Ar,Bvr,Ber); m=train_attn_ae(Za,seed); e=attn_err(m,Ze)
        roc["attn_ae_rate_sdj"].append(roc_auc_score(is_t[sdj].astype(int),e[sdj]))
        # simple benign-Mahalanobis on source-rate (reference / our method)
        mu,P=fit_mahal(Za[bmask]); se=mahal(Ze,mu,P); base=mahal(Zv,mu,P)[yv==normal]
        roc["mahal_rate_sdj"].append(roc_auc_score(is_t[sdj].astype(int),se[sdj]))
        op["mahal_rate"].append(op_pts(base,se[sdj],is_t[sdj],is_n[sdj],ka[sdj],grid))
    print("=== Source-disjoint ROC (LOACO, mean\u00b1std) ===")
    for k in roc: v=np.array(roc[k]); print(f"  {k:18s} {v.mean():.3f}\u00b1{v.std():.3f}")
    print("\n=== Operating points @ source-disjoint ===")
    for name,key in [("attn_ae_pf","attn_ae_pf"),("mahal_rate (ours)","mahal_rate")]:
        print(f"  [{name}]")
        for f in grid:
            D=pd.DataFrame([o[f] for o in op[key]]).mean()
            print(f"    fpr={f:.2f} det={D['det']:.3f} fpr_normal={D['fpr_normal']:.3f} fpr_known={D['fpr_known']:.3f}")
    return roc
RES=run()

=== Source-disjoint ROC (LOACO, mean±std) ===
  attn_ae_pf_sdj     0.337±0.052
  attn_ae_rate_sdj   0.996±0.005
  mahal_rate_sdj     0.907±0.002

=== Operating points @ source-disjoint ===
  [attn_ae_pf]
    fpr=0.05 det=0.015 fpr_normal=0.046 fpr_known=0.139
    fpr=0.10 det=0.023 fpr_normal=0.094 fpr_known=0.185
    fpr=0.20 det=0.041 fpr_normal=0.194 fpr_known=0.260
  [mahal_rate (ours)]
    fpr=0.05 det=0.786 fpr_normal=0.049 fpr_known=0.277
    fpr=0.10 det=0.925 fpr_normal=0.099 fpr_known=0.350
    fpr=0.20 det=1.000 fpr_normal=0.196 fpr_known=0.486


## Reading (pre-committed)

- If `attn_ae_pf_sdj` and `attn_ae_rate_sdj` are **below** `mahal_rate_sdj` (\u2248 0.907):
  **simple beats complex** \u2014 attention/capacity does not overcome the in-manifold limitation;
  the simple benign-anchored detector on the right representation wins. One row in Table 2 + one
  sentence in Discussion.
- If an attention variant **genuinely** exceeds it (source-disjoint, and not artifact-driven on the
  per-flow variant): report it honestly as a co-method.

Note: per-flow attention-AE may score higher on a non-source-disjoint test by exploiting collection
artifacts; we therefore compare on **source-disjoint** ROC, consistent with the rest of the study.

## Doğrulama

In [ ]:
# === attn_ae_rate doğrulama: operating points + permütasyon (null) kontrolü ===
def verify_attn_rate(seeds=SEEDS, grid=FPR_GRID, epochs=EPOCHS):
    cfg=UAV_CFG; idcfg=ID_CFG; lab=cfg["label_col"]; normal=cfg["normal"]; tgt=TARGET; src=idcfg["src"]
    df=load(cfg); df=pd.concat([df,source_rate_features(df,idcfg)],axis=1)
    op=[]; null_auc=[]; train_recon=[]
    for seed in seeds:
        tr,valk,test_=loaco_split(df,lab,tgt,seed,)
        test_=test_.reset_index(drop=True)
        ytr=tr[lab].astype(str).values; yv=valk[lab].astype(str).values; ye=test_[lab].astype(str).values
        is_t=(ye==tgt); is_n=(ye==normal); ka=(~is_t)&(~is_n); bmask=(ytr==normal)
        seen=is_t & test_[src].astype(str).isin(set(tr[src].astype(str))).values; sdj=~seen

        Ar=tr[RATE].reset_index(drop=True); Bvr=valk[RATE].reset_index(drop=True); Ber=test_[RATE].reset_index(drop=True)
        Za,Zv,Ze=scale(Ar,Bvr,Ber)

        # (1) gercek model: known-manifold uzerinde egit, operating points (kaynak-ayrik)
        m=train_attn_ae(Za,seed,epochs=epochs); e=attn_err(m,Ze)
        base=attn_err(m,Zv)[yv==normal]                      # benign-only validasyon esigi
        rec_tr=attn_err(m,Za).mean(); train_recon.append(rec_tr)
        opf={}
        for f in grid:
            thr=np.percentile(base,100*(1-f))
            opf[f]=dict(det=float(np.mean((e[sdj]>thr)[is_t[sdj]])),
                        fpr_normal=float(np.mean((e>thr)[is_n])),
                        fpr_known=float(np.mean((e>thr)[ka])))
        op.append(opf)

        # (2) NULL/permutasyon kontrolu: egitim satirlarini ozellik-bazinda karistir
        #     (her sutunu bagimsiz permute -> ozellikler-arasi yapi bozulur, marjinaller korunur)
        rng=np.random.default_rng(seed)
        Zp=Za.copy()
        for j in range(Zp.shape[1]):
            Zp[:,j]=Zp[rng.permutation(Zp.shape[0]),j]
        mp=train_attn_ae(Zp,seed,epochs=epochs); ep=attn_err(mp,Ze)
        null_auc.append(roc_auc_score(is_t[sdj].astype(int),ep[sdj]))

    print("=== attn_ae_rate operating points (source-disjoint, mean) ===")
    for f in grid:
        D=pd.DataFrame([o[f] for o in op]).mean()
        print(f"  fpr={f:.2f} -> det={D['det']:.3f}  fpr_normal={D['fpr_normal']:.3f}  fpr_known={D['fpr_known']:.3f}")
    na=np.array(null_auc)
    print(f"\n[null/permuted-train] attn_ae_rate ROC = {na.mean():.3f}\u00b1{na.std():.3f}")
    print(f"[diag] mean train reconstruction error = {np.mean(train_recon):.4f}")
    print("\nReading:")
    print(" - fpr_known dusuk/makul (<=~0.35) ve null ROC ~0.5  -> gercek ustunluk (co-method)")
    print(" - fpr_known yuksek (~0.5)                          -> trivial: her saldiriyi 'yeni' sayiyor")
    print(" - null ROC hala yuksek (>~0.8)                     -> kurulum/dusuk-boyut artefakti, sinyal degil")
    return op, null_auc

OP_RATE, NULL_RATE = verify_attn_rate()

=== attn_ae_rate operating points (source-disjoint, mean) ===
  fpr=0.05 -> det=1.000  fpr_normal=0.044  fpr_known=0.099
  fpr=0.10 -> det=1.000  fpr_normal=0.092  fpr_known=0.178
  fpr=0.20 -> det=1.000  fpr_normal=0.191  fpr_known=0.287

[null/permuted-train] attn_ae_rate ROC = 0.773±0.126
[diag] mean train reconstruction error = 0.0000

Reading:
 - fpr_known dusuk/makul (<=~0.35) ve null ROC ~0.5  -> gercek ustunluk (co-method)
 - fpr_known yuksek (~0.5)                          -> trivial: her saldiriyi 'yeni' sayiyor
 - null ROC hala yuksek (>~0.8)                     -> kurulum/dusuk-boyut artefakti, sinyal degil


# Gerçek Attention-AE bottleneck denemesi

In [ ]:
# === Yol 1: ADİL attention-AE (gerçek bottleneck: latent<girdi, daha az epoch) ===
# Mevcut TabAttnAE/attn_err/scale/load/... tanımlarını kullanır; hiçbirini değiştirmez.

def train_attn_ae_bottleneck(Xtr, seed, epochs=20, bs=512, lr=1e-3, latent=2):
    torch.manual_seed(seed)
    n=Xtr.shape[1]; d=32; nhead=4 if n>=4 else 1
    lat=min(latent, max(1, n-1))                 # latent GİRDİDEN KÜÇÜK (gerçek sıkıştırma)
    m=TabAttnAE(n, d=d, nhead=nhead, latent=lat).to(DEV)
    opt=torch.optim.Adam(m.parameters(), lr=lr)
    X=torch.tensor(Xtr, dtype=torch.float32)
    dl=torch.utils.data.DataLoader(torch.utils.data.TensorDataset(X), batch_size=bs, shuffle=True)
    lossf=nn.MSELoss(); m.train()
    for _ in range(epochs):
        for (xb,) in dl:
            xb=xb.to(DEV); opt.zero_grad()
            loss=lossf(m(xb), xb); loss.backward(); opt.step()
    return m, lat

def run_attn_rate_fair(seeds=SEEDS, grid=FPR_GRID, epochs=20, latent=2):
    cfg=UAV_CFG; idcfg=ID_CFG; lab=cfg["label_col"]; normal=cfg["normal"]; tgt=TARGET; src=idcfg["src"]
    df=load(cfg); df=pd.concat([df, source_rate_features(df, idcfg)], axis=1)
    roc=[]; op=[]; null_auc=[]; trec=[]; used_lat=None
    for seed in seeds:
        tr,valk,test_=loaco_split(df,lab,tgt,seed); test_=test_.reset_index(drop=True)
        ytr=tr[lab].astype(str).values; yv=valk[lab].astype(str).values; ye=test_[lab].astype(str).values
        is_t=(ye==tgt); is_n=(ye==normal); ka=(~is_t)&(~is_n)
        seen=is_t & test_[src].astype(str).isin(set(tr[src].astype(str))).values; sdj=~seen
        Ar=tr[RATE].reset_index(drop=True); Bvr=valk[RATE].reset_index(drop=True); Ber=test_[RATE].reset_index(drop=True)
        Za,Zv,Ze=scale(Ar,Bvr,Ber)
        # gercek bottleneck modeli
        m,lat=train_attn_ae_bottleneck(Za,seed,epochs=epochs,latent=latent); used_lat=lat
        e=attn_err(m,Ze); base=attn_err(m,Zv)[yv==normal]; trec.append(attn_err(m,Za).mean())
        roc.append(roc_auc_score(is_t[sdj].astype(int), e[sdj]))
        opf={}
        for f in grid:
            thr=np.percentile(base,100*(1-f))
            opf[f]=dict(det=float(np.mean((e[sdj]>thr)[is_t[sdj]])),
                        fpr_normal=float(np.mean((e>thr)[is_n])),
                        fpr_known=float(np.mean((e>thr)[ka])))
        op.append(opf)
        # null kontrolu (ayni bottleneck ile)
        rng=np.random.default_rng(seed); Zp=Za.copy()
        for j in range(Zp.shape[1]): Zp[:,j]=Zp[rng.permutation(Zp.shape[0]),j]
        mp,_=train_attn_ae_bottleneck(Zp,seed,epochs=epochs,latent=latent)
        null_auc.append(roc_auc_score(is_t[sdj].astype(int), attn_err(mp,Ze)[sdj]))
    r=np.array(roc); na=np.array(null_auc)
    print(f"=== attn_ae_rate (FAIR bottleneck: latent={used_lat}<3, epochs={epochs}) ===")
    print(f"  source-disjoint ROC = {r.mean():.3f}\u00b1{r.std():.3f}")
    print(f"  mean train recon error = {np.mean(trec):.4f}   (0.0000 ise hala sikismiyor demektir)")
    print(f"  null/permuted-train ROC = {na.mean():.3f}\u00b1{na.std():.3f}")
    print("  operating points (source-disjoint, mean):")
    for f in grid:
        D=pd.DataFrame([o[f] for o in op]).mean()
        print(f"    fpr={f:.2f} -> det={D['det']:.3f}  fpr_normal={D['fpr_normal']:.3f}  fpr_known={D['fpr_known']:.3f}")
    print("\n  Karsilastirma: mahal_rate_sdj (bizim) = 0.907 | onceki latent=8 attn = 0.996")
    print("  Okuma: ROC korunursa -> saglam co-method; belirgin duserse -> onceki 0.996 bottleneck-yoklugu etkisiydi")
    return roc, op, null_auc

In [ ]:
ROC_FAIR, OP_FAIR, NULL_FAIR = run_attn_rate_fair(epochs=20, latent=2)

=== attn_ae_rate (FAIR bottleneck: latent=2<3, epochs=20) ===
  source-disjoint ROC = 0.976±0.002
  mean train recon error = 0.0289   (0.0000 ise hala sikismiyor demektir)
  null/permuted-train ROC = 0.917±0.100
  operating points (source-disjoint, mean):
    fpr=0.05 -> det=0.973  fpr_normal=0.047  fpr_known=0.111
    fpr=0.10 -> det=0.973  fpr_normal=0.097  fpr_known=0.186
    fpr=0.20 -> det=0.978  fpr_normal=0.196  fpr_known=0.254

  Karsilastirma: mahal_rate_sdj (bizim) = 0.907 | onceki latent=8 attn = 0.996
  Okuma: ROC korunursa -> saglam co-method; belirgin duserse -> onceki 0.996 bottleneck-yoklugu etkisiydi


## Attention yorumlanması

In [ ]:
# === Attention yorumlanabilirligi: hangi ozelliklere agirlik veriliyor? ===
# attn_ae_rate (latent=2) uzerinde calisir; mevcut tanimlari kullanir.

import torch

def attn_feature_importance(seed=RS, epochs=20, latent=2):
    cfg=UAV_CFG; idcfg=ID_CFG; lab=cfg["label_col"]; normal=cfg["normal"]; tgt=TARGET; src=idcfg["src"]
    df=load(cfg); df=pd.concat([df, source_rate_features(df, idcfg)], axis=1)
    tr,valk,test_=loaco_split(df,lab,tgt,seed); test_=test_.reset_index(drop=True)
    ytr=tr[lab].astype(str).values; ye=test_[lab].astype(str).values
    is_t=(ye==tgt); is_n=(ye==normal)
    seen=is_t & test_[src].astype(str).isin(set(tr[src].astype(str))).values; sdj=~seen
    Ar=tr[RATE].reset_index(drop=True); Ber=test_[RATE].reset_index(drop=True)
    Za,_,Ze=scale(Ar,Ar,Ber)
    m,lat=train_attn_ae_bottleneck(Za,seed,epochs=epochs,latent=latent)
    m.eval()

    # (A) Attention agirliklari: ilk encoder katmanindan, Sybil vs normal test orneklerinde
    #     TransformerEncoderLayer self-attn agirligini almak icin forward hook
    attn_store={}
    layer=m.encoder.layers[0].self_attn
    def hook(mod, inp, out):
        # MultiheadAttention need_weights default; tekrar cagirarak agirlik al
        pass
    # Daha guvenilir: tokenlari elle gecirip attn agirligini iste
    with torch.no_grad():
        def tokens(x):
            B=x.size(0)
            return m.val_proj(x.unsqueeze(-1)) + m.feat_emb.unsqueeze(0)
        def layer_attn(tok):
            # ilk katmanin self-attn agirligi: [B, n, n] (head-ortalama)
            _,w = layer(tok, tok, tok, need_weights=True, average_attn_weights=True)
            return w
        Xs=torch.tensor(Ze[sdj][is_t[sdj]][:2000],dtype=torch.float32).to(DEV)
        Xn=torch.tensor(Ze[is_n][:2000],dtype=torch.float32).to(DEV)
        Wsyb=layer_attn(tokens(Xs)).mean(0).cpu().numpy()   # [3,3]
        Wnorm=layer_attn(tokens(Xn)).mean(0).cpu().numpy()
    # her ozelligin ALDIGI toplam dikkat (sutun toplami) = ne kadar "okunuyor"
    recv_syb=Wsyb.mean(0); recv_norm=Wnorm.mean(0)

    # (B) Permutasyon-onem: her ozelligi test'te boz, ROC ne kadar dusuyor?
    base=roc_auc_score(is_t[sdj].astype(int), attn_err(m,Ze)[sdj])
    rng=np.random.default_rng(0); imp=[]
    for j in range(len(RATE)):
        Zd=Ze.copy(); Zd[:,j]=Zd[rng.permutation(Zd.shape[0]),j]
        imp.append(base - roc_auc_score(is_t[sdj].astype(int), attn_err(m,Zd)[sdj]))

    print(f"base source-disjoint ROC = {base:.3f}\n")
    print(f"{'feature':20s} {'attn_recv_norm':>14} {'attn_recv_syb':>14} {'perm_importance':>16}")
    for j,f in enumerate(RATE):
        print(f"{f:20s} {recv_norm[j]:>14.3f} {recv_syb[j]:>14.3f} {imp[j]:>16.3f}")
    print("\nOkuma:")
    print(" - perm_importance yuksek olan ozellik = ROC icin kritik (modelin gercekten kullandigi)")
    print(" - tek bir ozellik baskinsa -> null-kontrolu (marjinal-agirlikli) ile tutarli")
    print(" - attn_recv: hangi ozellik token'i digerlerince cok 'okunuyor' (baglamsal rol)")
    return recv_norm, recv_syb, imp

FI = attn_feature_importance()

base source-disjoint ROC = 0.974

feature              attn_recv_norm  attn_recv_syb  perm_importance
s_fanout_rate                 0.344          0.734            0.080
s_flows_per_dst               0.385          0.136            0.302
s_dst_entropy_norm            0.271          0.130            0.353

Okuma:
 - perm_importance yuksek olan ozellik = ROC icin kritik (modelin gercekten kullandigi)
 - tek bir ozellik baskinsa -> null-kontrolu (marjinal-agirlikli) ile tutarli
 - attn_recv: hangi ozellik token'i digerlerince cok 'okunuyor' (baglamsal rol)


# csv'ye yaz

In [ ]:
# === Fig.2 ROC curve -> fig2_roc.csv (AttentionBaseline: attn_rate) ===
import os, numpy as np, pandas as pd
from sklearn.metrics import roc_curve

def fig2_save_attn(seeds=SEEDS, csv="fig2_roc.csv", epochs=20, latent=2):
    grid=np.linspace(0,1,101)
    cfg=UAV_CFG; lab=cfg["label_col"]; normal=cfg["normal"]; tgt=TARGET; src=ID_CFG["src"]
    df=load(cfg); df=pd.concat([df,source_rate_features(df,ID_CFG)],axis=1)
    curves=[]
    for seed in seeds:
        tr,valk,test_=loaco_split(df,lab,tgt,seed); test_=test_.reset_index(drop=True)
        ye=test_[lab].astype(str).values; is_t=(ye==tgt)
        seen=is_t & test_[src].astype(str).isin(set(tr[src].astype(str))).values; sdj=~seen
        Ar=tr[RATE].reset_index(drop=True); Ber=test_[RATE].reset_index(drop=True)
        Za,_,Ze=scale(Ar,Ar,Ber)
        m,_=train_attn_ae_bottleneck(Za,seed,epochs=epochs,latent=latent)
        e=attn_err(m,Ze)
        f,t,_=roc_curve(is_t[sdj].astype(int), e[sdj]); curves.append(np.interp(grid,f,t))
    A=np.vstack(curves)
    new=pd.DataFrame([{"detector":"attn_rate","fpr":float(x),"tpr_mean":float(mn),"tpr_std":float(sd)}
                      for x,mn,sd in zip(grid,A.mean(0),A.std(0))])
    if os.path.exists(csv):
        old=pd.read_csv(csv); old=old[old["detector"]!="attn_rate"]; new=pd.concat([old,new],ignore_index=True)
    new.to_csv(csv,index=False); print(f"wrote {csv} | detectors now:", sorted(new.detector.unique()))
fig2_save_attn()

wrote fig2_roc.csv | detectors now: ['ae_pf', 'attn_rate', 'mahal_pf', 'mahal_rate']
